# Análise Exploratória de Dados — Bank Personal Loan Modelling

**Projeto**: Plataforma de Experimentação Adaptativa — Datathon 7MLET  
**Dataset**: Bank Personal Loan Modelling (Kaggle)  
**Objetivo**: Entender a base, identificar variáveis relevantes, documentar
decisões de pré-processamento e justificar o uso no contexto de Multi-Armed Bandit.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

# Configurações de visualização
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

print("Bibliotecas carregadas com sucesso.")

## 1. Carregamento e Visão Geral

In [ ]:
# Carregamento do dataset
df = pd.read_csv('../data/kaggle/Bank_Personal_Loan_Modelling.csv')

print(f"Dimensões: {df.shape[0]} linhas x {df.shape[1]} colunas")
print(f"\nPrimeiras linhas:")
df.head()

In [ ]:
print("=== Tipos de dados e valores não-nulos ===")
df.info()

In [ ]:
print("=== Estatísticas Descritivas ===")
df.describe().round(2)

## 2. Dicionário de Dados

| Coluna | Tipo | Descrição | Uso no Projeto |
|--------|------|-----------|----------------|
| ID | int | Identificador único do cliente | Descartada |
| Age | int | Idade do cliente (anos) | Contexto do bandit |
| Experience | int | Anos de experiência profissional | Contexto do bandit (após limpeza) |
| Income | int | Renda mensal (unidades abstratas) | Contexto do bandit + suitability |
| ZIP Code | int | CEP do cliente | Descartada |
| Family | int | Tamanho do núcleo familiar (1-4) | Contexto do bandit |
| CCAvg | float | Gasto médio mensal no cartão (unidades) | Não utilizada |
| Education | int | Nível de escolaridade (1=grad, 2=pós, 3=avançado) | Contexto do bandit |
| Mortgage | int | Valor do financiamento imobiliário | Não utilizada |
| Personal Loan | int | **Target**: aceitou empréstimo pessoal (0/1) | Recompensa histórica |
| Securities Account | int | Possui conta de investimentos (0/1) | Contexto + suitability |
| CD Account | int | Possui certificado de depósito (0/1) | Contexto + suitability |
| Online | int | Usa internet banking (0/1) | Contexto do bandit |
| CreditCard | int | Possui cartão de crédito do banco (0/1) | Contexto + suitability |

## 3. Qualidade dos Dados

In [ ]:
print("=== Valores Nulos por Coluna ===")
nulos = df.isnull().sum()
pct_nulos = (nulos / len(df) * 100).round(2)
resumo_nulos = pd.DataFrame({'Nulos': nulos, '% Nulos': pct_nulos})
print(resumo_nulos[resumo_nulos['Nulos'] > 0] if resumo_nulos['Nulos'].sum() > 0 else "Nenhum valor nulo encontrado.")

In [ ]:
duplicados = df.duplicated().sum()
print(f"Registros duplicados: {duplicados}")
print(f"Registros únicos: {len(df) - duplicados}")

In [ ]:
print("=== Problema: Experience com valores negativos ===")
exp_negativo = df[df['Experience'] < 0]
print(f"Registros com Experience < 0: {len(exp_negativo)}")
print(f"Valores encontrados: {sorted(df['Experience'].unique())[:10]}")
print(f"\nDecisão: remover registros com Experience < 0 (valores inválidos)")
print(f"Registros restantes após limpeza: {len(df[df['Experience'] >= 0])}")

## 4. Análise do Target — Personal Loan

In [ ]:
# Distribuição do target
contagem = df['Personal Loan'].value_counts()
pct = df['Personal Loan'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Gráfico de barras
axes[0].bar(['Não aceitou (0)', 'Aceitou (1)'],
            contagem.values,
            color=['#e74c3c', '#2ecc71'])
axes[0].set_title('Distribuição do Target — Personal Loan')
axes[0].set_ylabel('Quantidade')
for i, v in enumerate(contagem.values):
    axes[0].text(i, v + 30, f'{v}\n({pct.values[i]:.1f}%)',
                ha='center', fontweight='bold')

# Pizza
axes[1].pie(contagem.values,
            labels=[f'Não aceitou\n{pct.values[0]:.1f}%',
                    f'Aceitou\n{pct.values[1]:.1f}%'],
            colors=['#e74c3c', '#2ecc71'],
            startangle=90)
axes[1].set_title('Proporção do Target')

plt.tight_layout()
plt.savefig('../reports/target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nDesbalanceamento: {pct.values[0]:.1f}% não aceitou vs {pct.values[1]:.1f}% aceitou")
print("Impacto: XGBoost treinado com scale_pos_weight para compensar desbalanceamento")

## 5. Análise das Variáveis de Contexto

In [ ]:
# Variáveis numéricas usadas no modelo
variaveis = ['Age', 'Experience', 'Income', 'Family', 'Education']

fig, axes = plt.subplots(1, 5, figsize=(18, 4))

for i, var in enumerate(variaveis):
    axes[i].hist(df[var], bins=20, color='#3498db', alpha=0.7, edgecolor='white')
    axes[i].set_title(var)
    axes[i].set_xlabel('Valor')
    if i == 0:
        axes[i].set_ylabel('Frequência')

plt.suptitle('Distribuição das Variáveis Numéricas de Contexto', y=1.02)
plt.tight_layout()
plt.savefig('../reports/numeric_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Variáveis binárias
binarias = ['Securities Account', 'CD Account', 'Online', 'CreditCard']

fig, axes = plt.subplots(1, 4, figsize=(14, 4))

for i, var in enumerate(binarias):
    contagem_var = df[var].value_counts()
    axes[i].bar(['Não (0)', 'Sim (1)'], contagem_var.values,
                color=['#e74c3c', '#2ecc71'])
    axes[i].set_title(var)
    axes[i].set_ylabel('Quantidade')
    for j, v in enumerate(contagem_var.values):
        axes[i].text(j, v + 20, str(v), ha='center', fontweight='bold')

plt.suptitle('Distribuição das Variáveis Binárias de Contexto', y=1.02)
plt.tight_layout()
plt.savefig('../reports/binary_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlação de cada variável com o target
colunas_modelo = ['Age', 'Experience', 'Income', 'Family', 'Education',
                  'Securities Account', 'CD Account', 'Online', 'CreditCard']

correlacoes = df[colunas_modelo + ['Personal Loan']].corr()['Personal Loan'].drop('Personal Loan')
correlacoes = correlacoes.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
cores = ['#e74c3c' if v < 0 else '#2ecc71' for v in correlacoes.values]
ax.barh(correlacoes.index, correlacoes.values, color=cores)
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_title('Correlação das Variáveis com Personal Loan (Target)')
ax.set_xlabel('Correlação de Pearson')
ax.grid(True, alpha=0.3, axis='x')

for i, v in enumerate(correlacoes.values):
    ax.text(v + 0.005 if v >= 0 else v - 0.005,
            i, f'{v:.3f}',
            va='center',
            ha='left' if v >= 0 else 'right',
            fontsize=9)

plt.tight_layout()
plt.savefig('../reports/correlations.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nVariáveis com maior correlação com o target:")
print(correlacoes.abs().sort_values(ascending=False).head(5))

## 6. Análise de Suitability

As regras de suitability foram definidas com base na análise das variáveis:

In [ ]:
df_clean = df[df['Experience'] >= 0].copy()

# Regra 1: Idade < 21
bloq_idade = (df_clean['Age'] < 21).sum()

# Regra 2: Income < 25
bloq_renda = (df_clean['Income'] < 25).sum()

# Regra 3: Sem relacionamento bancário
bloq_relacionamento = (
    (df_clean['CreditCard'] == 0) &
    (df_clean['CD Account'] == 0) &
    (df_clean['Securities Account'] == 0)
).sum()

# Combinado
mask_bloqueado = (
    (df_clean['Age'] < 21) |
    (df_clean['Income'] < 25) |
    ((df_clean['CreditCard'] == 0) &
     (df_clean['CD Account'] == 0) &
     (df_clean['Securities Account'] == 0))
)

total_bloqueado = mask_bloqueado.sum()
total_elegivel = (~mask_bloqueado).sum()

print("=== Impacto do Filtro de Suitability na Base ===\n")
print(f"Total de registros (após limpeza): {len(df_clean)}")
print(f"\nBloqueados por idade < 21:              {bloq_idade}")
print(f"Bloqueados por renda < 25:              {bloq_renda}")
print(f"Bloqueados por sem relacionamento:      {bloq_relacionamento}")
print(f"\nTotal bloqueado (pelo menos 1 regra):   {total_bloqueado} ({total_bloqueado/len(df_clean)*100:.1f}%)")
print(f"Total elegível:                         {total_elegivel} ({total_elegivel/len(df_clean)*100:.1f}%)")

# Taxa de conversão entre elegíveis
taxa_elegivel = df_clean[~mask_bloqueado]['Personal Loan'].mean() * 100
taxa_bloqueado = df_clean[mask_bloqueado]['Personal Loan'].mean() * 100
print(f"\nTaxa de conversão entre elegíveis:      {taxa_elegivel:.1f}%")
print(f"Taxa de conversão entre bloqueados:     {taxa_bloqueado:.1f}%")
print("\nConclusão: O filtro remove clientes com menor propensão, concentrando")
print("o sistema nos perfis com maior potencial de conversão.")

## 7. Colunas Descartadas e Justificativa

| Coluna | Motivo |
|--------|--------|
| ID | Identificador sem valor preditivo — vazamento de identidade |
| ZIP Code | Dado geográfico sensível sem valor preditivo para o modelo |
| CCAvg | Informação de gasto médio no cartão — não utilizada neste experimento |
| Mortgage | Valor do financiamento — não utilizado neste experimento |
| Experience (< 0) | Valores negativos inválidos — removidos da base |

**Nota sobre vazamento temporal**: O dataset não contém a variável `duration`
(presente no Bank Marketing UCI), que representaria o tempo de contato e
causaria vazamento temporal. A base utilizada não apresenta esse problema.

## 8. Conclusões e Decisões de Design

### Dataset escolhido
Bank Personal Loan Modelling — Kaggle
- 5000 clientes, 14 variáveis originais
- Target: aceitação de empréstimo pessoal (binário)
- Sem vazamento temporal identificado

### Variáveis selecionadas para o modelo (9 variáveis)
Age, Experience, Income, Family, Education, Securities Account,
CD Account, Online, CreditCard

### Decisões de pré-processamento
1. **Remoção de Experience < 0**: 52 registros com valores inválidos removidos
2. **Desbalanceamento**: tratado com `scale_pos_weight` no XGBoost
3. **Suitability**: 3 regras de elegibilidade baseadas em análise das variáveis

### Conexão com o Multi-Armed Bandit
- As 9 variáveis formam o **vetor de contexto** de cada decisão
- O **target** (Personal Loan) serve como base para a recompensa histórica
- O desbalanceamento (~9.6% de conversão) motivou a inicialização
  balanceada do bandit (50/50) em vez de usar as previsões do XGBoost diretamente

In [ ]:
# Salvar base processada (sem colunas desnecessárias, sem Experience negativo)
df_processed = df[df['Experience'] >= 0][
    ['Age', 'Experience', 'Income', 'Family', 'Education',
     'Securities Account', 'CD Account', 'Online', 'CreditCard', 'Personal Loan']
].copy()

df_processed.to_csv('../data/processed/bank_loan_processed.csv', index=False)

print(f"Base processada salva em data/processed/bank_loan_processed.csv")
print(f"Dimensões: {df_processed.shape[0]} linhas x {df_processed.shape[1]} colunas")
print(f"\nPrimeiras linhas:")
df_processed.head()